# Week 6 — Re-evaluate the supplied CNN checkpoints

This notebook is a **new replay helper**, not the original experimental record. It has no pre-filled result outputs. It uses the two checkpoints in this package and adds a clean-model trigger control that was absent from the submitted evaluation.

The original Colab notebook, including its saved outputs, is `01_original_experiment.ipynb`. Original reported values are **71.69% / 72.32% / 39.48%**; the code below computes fresh values instead of copying them.

Select a GPU runtime before setup when available. The optional network demonstration uses a fixed fictional payload and a loopback receiver only. No remote shell or secret collection is implemented.

## 1. Install the dataset helper

Colab normally provides PyTorch. This cell installs the Hugging Face dataset package. Package installation and the dataset download require network access; they are separate from the loopback-only demonstration.

In [ ]:
%pip -q install datasets

## 2. Upload and extract the complete project ZIP

Select the **final Week 6 package ZIP**, not an individual checkpoint. Extraction validates paths before writing. Rerunning this cell in the same runtime reuses the extracted directory.

In [ ]:
from pathlib import Path
import zipfile
import sys

BASE = Path("/content/Week6_Package")
PROJECT = BASE / "week6"

if not (PROJECT / "src/evaluate.py").is_file():
    from google.colab import files
    uploaded = files.upload()
    candidates = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(candidates) != 1:
        raise ValueError("Upload exactly one final Week 6 project ZIP.")
    BASE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(candidates[0]) as archive:
        for member in archive.infolist():
            destination = (BASE / member.filename).resolve()
            if not destination.is_relative_to(BASE.resolve()):
                raise ValueError("Archive has an unsafe path.")
        archive.extractall(BASE)

required = ["models/clean_model.pth", "models/backdoored_model.pth", "src/evaluate.py"]
for relative in required:
    if not (PROJECT / relative).is_file():
        raise FileNotFoundError(f"Missing {relative}; use the complete final package.")

sys.path.insert(0, str(PROJECT / "src"))
from verify_package import verify
assert verify(), "A packaged source file differs from its manifest."
print("Project ready:", PROJECT)

## 3. Check the current runtime

This is the runtime for the new replay, not a claim about the original run.

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 4. Recompute metrics using the supplied weights

The evaluation reports original-image accuracy for both models, targeted ASR excluding true airplane inputs, and the **clean-model target rate on triggered inputs**. The latter is an added control. Per-image predictions, raw counts and environment details are written to a new `runs/evaluation_...` directory.

The supplied `data/results.csv` remains unchanged.

In [ ]:
from evaluate import run_evaluation
new_results = run_evaluation(device_name="auto", workers=0)

## 5. Optional selected example and loopback HTTP action

The search chooses the first eligible successful case in the first 500 test rows. This is an illustrative selection, not a representative sample. The helper records the selected index and checks both models.

With `send_demo=True`, a separate Python function sends only a fixed fictional dictionary to `127.0.0.1`. It does not read company files, contact an external host or open a reverse shell. Set it to `False` to inspect model predictions without sending the request.

In [ ]:
from demo import run_demo
new_demo = run_demo(device_name="auto", limit=500, send_demo=True)

## 6. Download the new outputs

This ZIP contains only new run outputs. Do not replace the historical notebook outputs or paste the old expected numbers into a new result table.

In [ ]:
from google.colab import files
archive_path = BASE / "Week6_New_Run_Outputs.zip"
runs = PROJECT / "runs"
if not runs.is_dir():
    raise FileNotFoundError("No run outputs exist yet; complete an evaluation or demo first.")
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(runs.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=str(path.relative_to(PROJECT)))
files.download(str(archive_path))

## Interpretation reminder

A selected target-class change is not a general backdoor detector. Compare aggregate controls before making causal claims. The CNN only returns scores; the HTTP action is explicitly implemented outside the model.

Read `README.md`, `docs/mitre_mapping.md` and `evidence/provenance.md` for the submitted experiment and its limits.